In [1]:
import sys

sys.path.insert(0, "/workspace/git_version_clear")
from src.transformations.gold_transformations import sales_and_revenues, geography, customer_metrics, products, orders, customers_loyalty

from pyspark.testing import assertDataFrameEqual
from pyspark.sql.functions import col
from pyspark.sql import SparkSession
from pyspark.sql.types import DayTimeIntervalType, StructType, StructField, TimestampType, FloatType, IntegerType, DateType, DoubleType, LongType, BooleanType, StringType


import src.utils.logger as L

logger = L.logging.getLogger("logger")

In [2]:
spark_session = (
        SparkSession.builder
        .appName("test")
        .master("local[*]")
        .config("spark.python.worker.faulthandler.enabled", "true")
        .config("spark.sql.execution.pyspark.udf.faulthandler.enabled", "true")
        .getOrCreate()
        )

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/12 19:47:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [12]:
import pyspark.sql.functions as sf
columns = ["id", "name", "age", "country"]

data = [["1", "Adam", "25", "PL"],
        ["2", "Anna", "17", "PL"],
        ["3", "John", "31", "UK"],
        ["4", "Mark", "16", "UK"]]

dataframe = spark_session.createDataFrame(data, columns)
dataframe = (dataframe
             .select(dataframe.id.cast("integer").alias("id"), 
                     dataframe.name.cast("string").alias("name"),
                     dataframe.age.cast("integer").alias("age"),
                     dataframe.country.cast("string").alias("country")))

dataframe = (dataframe
             .filter(dataframe.age > 18)
             .withColumn("is_polish", sf.when(dataframe.country == "PL", True).otherwise(False)))

dataframe.show()

+---+----+---+-------+---------+
| id|name|age|country|is_polish|
+---+----+---+-------+---------+
|  1|Adam| 25|     PL|     true|
|  3|John| 31|     UK|    false|
+---+----+---+-------+---------+



In [21]:
columns = ["customer_id", "amount"]

data = [["1", "100"],
        ["1", "50"],
        ["2", "200"],
        ["1", "75"],
        ["2", "20"]]

dataframe = spark_session.createDataFrame(data, columns)
dataframe = (dataframe
             .select(dataframe.customer_id.cast("integer").alias("customer_id"), 
                     dataframe.amount.cast("integer").alias("amount")))


dataframe = (dataframe
             .groupBy(dataframe.customer_id)
             .agg(sf.sum(dataframe.amount).alias("total_amount"),
                  sf.count(dataframe.customer_id).alias("orders_count")))

dataframe.show()

+-----------+------------+------------+
|customer_id|total_amount|orders_count|
+-----------+------------+------------+
|          1|         225|           3|
|          2|         220|           2|
+-----------+------------+------------+



In [33]:
columns_customers = ["customer_id", "name"]

data_customers = [["1", "Anna"],
                    ["2", "Tomasz"],
                    ["3", "Karol"],
                    ["4", "Wiktor"],
                    ["5", "Michal"]]

columns_orders = ["order_id", "customer_id", "amount"]
data_orders = [["1", "1", "300"],
                ["2", "2", "400"],
                ["3", "3", "500"],
                ["4", "4", "600"],
                ["4", "4", "700"]]

dataframe_customers = spark_session.createDataFrame(data_customers, columns_customers)
dataframe_orders = spark_session.createDataFrame(data_orders, columns_orders)

dataframe_customers = (dataframe_customers
                  .withColumn("customer_id", dataframe_customers.customer_id.cast("integer")))

dataframe_orders = (dataframe_orders
                  .withColumn("order_id", dataframe_orders.order_id.cast("integer"))
                  .withColumn("customer_id", dataframe_orders.customer_id.cast("integer"))
                  .withColumn("amount", dataframe_orders.amount.cast("integer")))


joined_dataframe = (dataframe_customers
                    .join(other = dataframe_orders,
                          on = "customer_id",
                          how = "left"))

joined_dataframe.show()

+-----------+------+--------+------+
|customer_id|  name|order_id|amount|
+-----------+------+--------+------+
|          1|  Anna|       1|   300|
|          2|Tomasz|       2|   400|
|          3| Karol|       3|   500|
|          4|Wiktor|       4|   700|
|          4|Wiktor|       4|   600|
|          5|Michal|    NULL|  NULL|
+-----------+------+--------+------+



In [45]:
columns = ["id", "amount"]

data = [["1", "100.50"],
        ["2", "ABC"],
        ["3", None],
        ["4", "50.20"]]

dataframe = spark_session.createDataFrame(data, columns)

dataframe.show()

dataframe = (dataframe
             .withColumn("amount", sf.col("amount").try_cast("double"))
             .filter(sf.col("amount").isNotNull()))

dataframe.show()

collect = dataframe.collect()[0][1]
counteee = dataframe.count()

print(counteee)

print(collect)


+---+------+
| id|amount|
+---+------+
|  1|100.50|
|  2|   ABC|
|  3|  NULL|
|  4| 50.20|
+---+------+

+---+------+
| id|amount|
+---+------+
|  1| 100.5|
|  4|  50.2|
+---+------+

2
100.5


In [ ]:
def transform_customers(df):
    df = (df
          .filter(sf.col("email").isNotNull())
          .filter(sf.col("age") >= 18)
          .withColumn("full_name", sf.col("first_name") + " " + sf.col("last_name")))

    return df

def test_example():
    data = [["Adam", "Nowak", "test@test.com", 19]]

    columns = StructType([
                StructField("first_name", StringType(), True),
                StructField("last_name", StringType(), True),
                StructField("email", StringType(), True),
                StructField("age", IntegerType(), True),
                ])

    test_dataframe = spark_session.createDataFrame(data, columns)

    tested_final = transform_customers(test_dataframe)

    counter = tested_final.count()
    full_name = tested_final.collect()[0][4]

    assert counter == 1
    assert full_name == "Adam Nowak"

    

    

In [3]:
""" ORDERS """

raw_orders_data_path = "/workspace/git_version_clear/tests/sample_data/orders.csv"

raw_orders_schema = StructType([
                StructField("order_id", IntegerType(), True),
                StructField("user_id", StringType(), True),
                StructField("order_date", TimestampType(), True),
                StructField("total_amount", FloatType(), True),
                ])

raw_orders_dataframe = spark_session.read.csv(path = raw_orders_data_path, schema = raw_orders_schema, header = True)

""" USERS """

raw_users_data_path = "/workspace/git_version_clear/tests/sample_data/users.csv"


raw_users_schema = StructType([
                    StructField("user_id", StringType(), True),
                    StructField("name", StringType(), True),
                    StructField("email", StringType(), True),
                    StructField("gender", StringType(), True),
                    StructField("city", StringType(), True),
                    StructField("signup_date", DateType(), True)
                    ])

raw_users_dataframe = spark_session.read.csv(path = raw_users_data_path, schema = raw_users_schema, header = True)

""" ORDER ITEMS """

raw_order_items_data_path = "/workspace/git_version_clear/tests/sample_data/order_items.csv"

raw_order_items_schema = StructType([
                        StructField("order_item_id", StringType(), True),
                        StructField("order_id", StringType(), True),
                        StructField("product_id", StringType(), True),
                        StructField("user_id", StringType(), True),
                        StructField("quantity", IntegerType(), True),
                        StructField("item_price", DoubleType(), True),
                        StructField("item_total", DoubleType(), True)
                        ])

raw_order_items_dataframe = spark_session.read.csv(path = raw_order_items_data_path, schema = raw_order_items_schema, header = True)



In [ ]:
expected_final_schema = StructType([
                        StructField("order_day", DateType(), True),
                        StructField("daily_revenue", DoubleType(), True),
                        StructField("avg_total_amount", DoubleType(), True),
                        StructField("order_count", LongType(), True),
                        StructField("revenue_ranking", IntegerType(), True),
                        StructField("is_in_top_5", BooleanType(), True),
                        StructField("avg_total_amount_change", DoubleType(), True),
                        StructField("pct_change_rate", DoubleType(), True),
])

expected_final_path = "/workspace/git_version_clear/tests/expected_data/sales_and_revenues/expected_data.csv"

expected_final_dataframe = spark_session.read.csv(path = expected_final_path, schema = expected_final_schema, header = False)

final_dataframe, total_system_revenue = sales_and_revenues(raw_orders_dataframe)
#final_dataframe.write.csv("/workspace/git_version_clear/tests/expected_data/sales_and_revenues_expected.csv")

print("raw")
raw_orders_dataframe.show(5)
print("expected_final")
expected_final_dataframe.show(5)
print("final")
final_dataframe.show(5)

#print(te_same)

print("Sprawdzanie")
if final_dataframe == expected_final_dataframe:
    print("taaaaaaaaaaaa")
#final_dataframe.show()
#final_dataframe.printSchema()
print(total_system_revenue)

In [ ]:
EXPECTED_SHARE_IN_TOTAL_REVENUE = 50.0
EXPECTED_FINAL_PATH = "/workspace/git_version_clear/tests/expected_data/geography/expected_data.csv"
SAMPLE_TOTAL_SYSTEM_REVENUE = 420.0

expected_final_schema = StructType([
                                StructField("city", StringType(), True),
                                StructField("revenue_per_city", DoubleType(), True),
                                StructField("users_count", LongType(), True),
                                StructField("avg_per_user", DoubleType(), True),
                                StructField("revenue_per_city_ranking", IntegerType(), True),
                                StructField("users_count_ranking", IntegerType(), True)
                            ])


final_geo_dataframe, test_share_in_total = geography(orders_dataframe = raw_orders_dataframe, users_dataframe = raw_users_dataframe, total_system_revenue = SAMPLE_TOTAL_SYSTEM_REVENUE, logger = logger)

#final_geo_dataframe.write.csv("/workspace/git_version_clear/tests/expected_data/geography")

expected_final_dataframe = spark_session.read.csv(path = EXPECTED_FINAL_PATH, schema = expected_final_schema, header = False)

final_geo_dataframe.show()
expected_final_dataframe.show()

In [ ]:
expected_final_schema = StructType([
                                StructField("user_id", StringType(), True),
                                StructField("summed_amount", DoubleType(), True),
                                StructField("orders_count", LongType(), True),
                                StructField("spent_segment", StringType(), True),
                                StructField("more_than_1_order", BooleanType(), True)
                            ])

final_customer_metrics_dataframe, active_users_count, avg_orders_count_per_user = customer_metrics(raw_orders_dataframe)

#final_customer_metrics_dataframe.write.csv("/workspace/git_version_clear/tests/expected_data/customer_metrics")
expected_final_dataframe = spark_session.read.csv(path = "/workspace/git_version_clear/tests/expected_data/customer_metrics/expected_data.csv", schema = expected_final_schema, header = False)
print("exected")
expected_final_dataframe.show(5)
print("final")
final_dataframe.show(5)
final_dataframe.printSchema()

In [ ]:
expected_final_gold_schema = StructType([
                                StructField("product_id", StringType(), True),
                                StructField("sold_count", LongType(), True),
                                StructField("summary_revenue", DoubleType(), True),
                                StructField("sold_count_ranking", IntegerType(), True),
                                StructField("top_revenue_ranking", IntegerType(), True)
                            ])

expected_final_top_prod_schema = StructType([
                                StructField("product_id", StringType(), True),
                                StructField("city", StringType(), True),
                                StructField("ordered_quantity", LongType(), True),
                                StructField("product_per_city_ranking", IntegerType(), True)
                            ])

final_dataframe, top_products_per_city_dataframe = products(raw_order_items_dataframe, raw_users_dataframe)
#final_dataframe.show()
#top_products_per_city_dataframe.show()

expected_final_dataframe = spark_session.read.csv(path = "/workspace/git_version_clear/tests/expected_data/products/gold_expected.csv", 
                                                  schema = expected_final_gold_schema, 
                                                  header = False)
expected_final_top_products_dataframe = spark_session.read.csv(path = "/workspace/git_version_clear/tests/expected_data/products/top_products_per_city.csv", 
                                                  schema = expected_final_top_prod_schema, 
                                                  header = False)

#final_dataframe.show()
#expected_final_dataframe.show()

top_products_per_city_dataframe.show()
expected_final_top_products_dataframe.show()

In [ ]:
expected_final_schema = StructType([
                                StructField("order_id", IntegerType(), True),
                                StructField("total_amount", FloatType(), True),
                                StructField("bucket", StringType(), True)
                            ])

final_dataframe, big_orders_share = orders(orders_dataframe = raw_orders_dataframe, 
                                           order_items_dataframe = raw_order_items_dataframe, 
                                           logger = logger)

final_dataframe.printSchema()

In [ ]:
expected_final_schema = StructType([
                                StructField("user_id", StringType(), True),
                                StructField("order_id", IntegerType(), True),
                                StructField("order_date", TimestampType(), True),
                                StructField("total_amount", FloatType(), True),
                                StructField("orders_count", LongType(), True),
                                StructField("user_total_amount", DoubleType(), True),
                                StructField("prev_order_date", TimestampType(), True),
                                StructField("orders_interval", DayTimeIntervalType(), True),
                                StructField("avg_interval", DayTimeIntervalType(), True),
                                StructField("loyalty_ranking", IntegerType(), True),
                            ])

final_dataframe = customers_loyalty(raw_orders_dataframe)

final_dataframe.printSchema()
#final_dataframe.show()

root
 |-- user_id: string (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- total_amount: float (nullable = true)
 |-- orders_count: long (nullable = false)
 |-- user_total_amount: double (nullable = true)
 |-- prev_order_date: timestamp (nullable = true)
 |-- orders_interval: interval day to second (nullable = true)
 |-- avg_interval: interval day to second (nullable = true)
 |-- loyalty_ranking: integer (nullable = false)

